## 1. Configuración del Entorno y Librerías
Importamos las herramientas necesarias para la manipulación de datos (`pandas`, `numpy`), el modelado predictivo (`scikit-learn`) y conectamos el entorno con Google Drive para acceder a las bases de datos.

In [ ]:
#==========================================================
# BLOQUE 1
# MONTAJE DE GOOGLE DRIVE Y LIBRERÍAS
#==========================================================

from google.colab import drive
drive.mount('/content/drive')
#==========================================================
# LIBRERÍAS
#==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

Mounted at /content/drive


## 2. Carga de Datos
Ingestamos los diferentes datasets que componen nuestro universo de información: datos históricos de partidos, estadísticas consolidadas por equipo, grupos del mundial y rankings de la FIFA.

In [ ]:
#==========================================================
# BLOQUE 2
# CARGAR ARCHIVOS
#==========================================================

base_path = "/content/drive/MyDrive/Simulaciones_Mundial/Data"

datos_mundial = pd.read_csv(f"{base_path}/datos_mundial.csv")

datos_historicos = pd.read_csv(f"{base_path}/datos_historicos.csv")

partidos = pd.read_csv(f"{base_path}/partidos.csv")

grupos = pd.read_csv(f"{base_path}/Grupos_Mundial.csv", sep=';')

ranking = pd.read_csv(f"{base_path}/ranking_fifa.csv")

transfermarkt = pd.read_csv(f"{base_path}/transfermarkt.csv")
print("datos_mundial:", datos_mundial.shape)
print("datos_historicos:", datos_historicos.shape)
print("partidos:", partidos.shape)
print("grupos:", grupos.shape)
print("ranking:", ranking.shape)
print("transfermarkt:", transfermarkt.shape)

print(datos_mundial.columns.tolist()[:20])

print()

print(datos_historicos.columns.tolist()[:20])

datos_mundial: (48, 47)
datos_historicos: (1396, 45)
partidos: (3316, 2778)
grupos: (48, 2)
ranking: (6931, 3)
transfermarkt: (217, 2)
['Equipo', 'Puntos', 'Peso', 'Valor_Mercado_Millones_Eur', 'Tipo_Equipo', 'Fecha', 'Resultado_1X2', 'avg_Goles_esperados_(xG)_total', 'avg_Posesión_total', 'avg_Remates_a_puerta_total', 'avg_Córneres_total', 'avg_Tarjetas_amarillas_total', 'avg_Faltas_total', 'avg_Paradas_total', 'avg_Pases_Pct_total', 'avg_Pases_Exitosos_total', 'avg_Goles_total', 'avg_Ptos_total', 'avg_xG_Share_total', 'avg_Remates_Puerta_Share_total']

['Fecha', 'Equipo_Local', 'Equipo_Visitante', 'Goles_Local', 'Goles_Visitante', 'Peso_Local', 'avg_Goles_esperados_(xG)_total_Local', 'avg_Tarjetas_amarillas_total_Local', 'avg_Faltas_total_Local', 'avg_Remates_a_puerta_3_Local', 'avg_Córneres_3_Local', 'avg_Tarjetas_amarillas_3_Local', 'avg_Faltas_3_Local', 'avg_Paradas_3_Local', 'avg_Pases_Pct_3_Local', 'avg_Pases_Exitosos_3_Local', 'avg_Paradas_Share_3_Local', 'Peso_Visitante', 'Res

In [ ]:
# Elimina espacios en los nombres de las columnas
datos_mundial.columns = datos_mundial.columns.str.strip()
grupos.columns = grupos.columns.str.strip()
datos_historicos.columns = datos_historicos.columns.str.strip()
partidos.columns = partidos.columns.str.strip()

## 3. Normalización y Limpieza de Datos
Para que el modelo pueda cruzar la información correctamente, es vital estandarizar los nombres de los países. Aplicamos un diccionario de mapeo para corregir variaciones en los nombres (ej. "USA" -> "Estados Unidos").

In [ ]:
#==========================================================
# BLOQUE 3
# NORMALIZACIÓN DE NOMBRES DE EQUIPOS
#==========================================================

# Diccionario de equivalencias
normalizar_equipo = {

    # América
    "USA": "Estados Unidos",
    "United States": "Estados Unidos",

    # Europa
    "England": "Inglaterra",
    "Netherlands": "Países Bajos",
    "Holland": "Países Bajos",
    "Switzerland": "Suiza",
    "Belgium": "Bélgica",
    "Germany": "Alemania",
    "Spain": "España",
    "Portugal": "Portugal",
    "France": "Francia",
    "Austria": "Austria",
    "Sweden": "Suecia",
    "Norway": "Noruega",
    "Bosnia & Herzegovina": "Bosnia y Herzegovina",
    "Bosnia and Herzegovina": "Bosnia y Herzegovina",

    # África
    "Ivory Coast": "Costa de Marfil",
    "South Africa": "Sudáfrica",
    "Cape Verde": "Cabo Verde",
    "Egypt": "Egipto",
    "Ghana": "Ghana",
    "Morocco": "Marruecos",

    # Asia
    "South Korea": "Corea del Sur",
    "Korea Republic": "Corea del Sur",
    "Japan": "Japón",
    "Iran": "Irán",

    # Oceanía
    "Australia": "Australia",

    # Sudamérica
    "Brazil": "Brasil",
    "Argentina": "Argentina",
    "Colombia": "Colombia",
    "Paraguay": "Paraguay",
    "Ecuador": "Ecuador",

    # Norteamérica
    "Canada": "Canadá",
    "Mexico": "México",
    "Senegal": "Senegal"
}

def normalizar_nombre(nombre):
    """
    Devuelve el nombre normalizado de un equipo.
    Si no existe en el diccionario, devuelve el original.
    """
    if pd.isna(nombre):
        return nombre

    nombre = str(nombre).strip()

    return normalizar_equipo.get(nombre, nombre)

    # datos_mundial
datos_mundial["Equipo"] = datos_mundial["Equipo"].apply(normalizar_nombre)

# históricos
datos_historicos["Equipo_Local"] = datos_historicos["Equipo_Local"].apply(normalizar_nombre)

datos_historicos["Equipo_Visitante"] = datos_historicos["Equipo_Visitante"].apply(normalizar_nombre)

# grupos
grupos['Equipo'] = grupos['Equipo'].apply(normalizar_nombre)

In [ ]:
print('Las columnas actuales del DataFrame `grupos` son:')
print(grupos.columns.tolist())

Las columnas actuales del DataFrame `grupos` son:
['Equipo', 'Grupo']


## 4. Ingeniería de Características (Feature Engineering)
Separamos las variables predictoras (features) de los equipos locales y visitantes. Nuestro objetivo (target) será predecir los `Goles_Local` y `Goles_Visitante` basándonos en métricas de rendimiento acumuladas.

In [ ]:
#==========================================================
# BLOQUE 4
# CONSTRUCCIÓN AUTOMÁTICA DE FEATURES
#==========================================================

# Variables Local
features_local = [
    c for c in datos_historicos.columns
    if c.startswith("avg_") and c.endswith("_Local")
]

# Variables Visitante
features_visit = [
    c for c in datos_historicos.columns
    if c.startswith("avg_") and c.endswith("_Visitante")
]

print("Variables Local:", len(features_local))
print("Variables Visitante:", len(features_visit))

# Unimos todas las disponibles
X = datos_historicos[features_local + features_visit].copy()

# Targets
y_local = datos_historicos["Goles_Local"]
y_visitante = datos_historicos["Goles_Visitante"]

print("\nDimensiones de X:", X.shape)

Variables Local: 11
Variables Visitante: 13

Dimensiones de X: (1396, 24)


In [ ]:
#==========================================================
# LIMPIEZA
#==========================================================

# Eliminar partidos sin goles registrados
datos_validos = datos_historicos.dropna(
    subset=["Goles_Local", "Goles_Visitante"]
).copy()

# Reconstruir X e y con los datos válidos
X = datos_validos[features_local + features_visit]

y_local = datos_validos["Goles_Local"]
y_visitante = datos_validos["Goles_Visitante"]

# Rellenar valores faltantes con la media
X = X.fillna(X.mean(numeric_only=True))

print("X:", X.shape)
print("Nulos:", X.isnull().sum().sum())

X: (1394, 24)
Nulos: 0


## 5. Preparación del Modelo (Train/Test Split)
Dividimos nuestro conjunto de datos históricos: 80% para que el modelo aprenda y 20% para evaluar su rendimiento. Además, aplicamos `StandardScaler` para que todas las variables numéricas tengan el mismo peso estadístico.

In [ ]:
#==========================================================
# BLOQUE 5
# DIVISIÓN DEL CONJUNTO DE DATOS Y ESCALADO
#==========================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# División entrenamiento/prueba
X_train, X_test, y_train_local, y_test_local = train_test_split(
    X,
    y_local,
    test_size=0.20,
    random_state=42
)

_, _, y_train_visit, y_test_visit = train_test_split(
    X,
    y_visitante,
    test_size=0.20,
    random_state=42
)

# Inicializar y ajustar el StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)
print("✅ StandardScaler inicializado y ajustado.")

Entrenamiento: (1115, 24)
Prueba: (279, 24)
✅ StandardScaler inicializado y ajustado.


In [ ]:
#==========================================================
# RANDOM FOREST
#==========================================================

from sklearn.ensemble import RandomForestRegressor

rf_local = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_visit = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_local.fit(X_train, y_train_local)
rf_visit.fit(X_train, y_train_visit)

print("✅ Modelos entrenados correctamente.")

✅ Modelos entrenados correctamente.


In [ ]:
pred_local = rf_local.predict(X_test)
pred_visit = rf_visit.predict(X_test)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("========== LOCAL ==========")

print("MAE :", mean_absolute_error(y_test_local, pred_local))
print("RMSE:", np.sqrt(mean_squared_error(y_test_local, pred_local)))
print("R²  :", r2_score(y_test_local, pred_local))

print()

print("======= VISITANTE ========")

print("MAE :", mean_absolute_error(y_test_visit, pred_visit))
print("RMSE:", np.sqrt(mean_squared_error(y_test_visit, pred_visit)))
print("R²  :", r2_score(y_test_visit, pred_visit))

========== LOCAL ==========
MAE : 1.0121492085937913
RMSE: 1.3966400233903893
R²  : 0.16735078061591224

======= VISITANTE ========
MAE : 0.942234163373392
RMSE: 1.2608378104238158
R²  : 0.2988885394640719
